In [8]:
import pandas as pd
import json
import os
from collections import Counter

In [9]:
df = pd.read_csv("../data/stack-overflow-developer-survey-2025/survey_results_public.csv", low_memory=False)
os.makedirs("../docs/data", exist_ok=True)

# P1 ¿En qué medida han adoptado los desarrolladores los AI Agents en 2025, y cómo varía esta
# adopción por país, experiencia, tipo de rol e industria?

# Por país (mapa)
adoption_country = (
    df[df["AIAgents"].notna() & df["Country"].notna()]
    .groupby("Country")["AIAgents"]
    .apply(lambda x: (x.str.startswith("Yes").sum() / len(x) * 100).round(1))
    .reset_index()
    .rename(columns={"AIAgents": "adoption_pct"})
)
adoption_country.to_json("../docs/data/p1_adoption_country.json", orient="records")

# Por DevType
adoption_devtype = (
    df[df["AIAgents"].notna() & df["DevType"].notna()]
    .assign(DevType=df["DevType"].str.split(";").str[0])
    .groupby("DevType")["AIAgents"]
    .apply(lambda x: (x.str.startswith("Yes").sum() / len(x) * 100).round(1))
    .reset_index()
    .rename(columns={"AIAgents": "adoption_pct"})
    .sort_values("adoption_pct", ascending=False)
    .head(15)
)
adoption_devtype.to_json("../docs/data/p1_adoption_devtype.json", orient="records")   

# Distribución general AIAgents
adoption_dist = df["AIAgents"].value_counts().reset_index()
adoption_dist.columns = ["category", "count"]
adoption_dist.to_json("../docs/data/p1_adoption_dist.json", orient="records")

# P2 P2 ¿Los desarrolladores que usan AI Agents reportan mayor o menor satisfacción laboral? ¿Existe
# relación entre adopción de IA y percepción de amenaza al empleo (AIThreat) o consideración de
# cambio de carrera (NewRole)?

# JobSat medio por nivel de uso de AIAgents
jobsat_agents = (
    df[df["AIAgents"].notna() & df["JobSat"].notna()]
    .groupby("AIAgents")["JobSat"]
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"mean": "jobsat_mean", "count": "n"})
)
jobsat_agents["jobsat_mean"] = jobsat_agents["jobsat_mean"].round(2)
jobsat_agents.to_json("../docs/data/p2_jobsat_agents.json", orient="records")

# AIThreat distribución
aithreat_dist = df["AIThreat"].value_counts().reset_index()
aithreat_dist.columns = ["category", "count"]
aithreat_dist.to_json("../docs/data/p2_aithreat.json", orient="records")

# NewRole distribución
newrole_dist = df["NewRole"].value_counts().reset_index()
newrole_dist.columns = ["category", "count"]
newrole_dist.to_json("../docs/data/p2_newrole.json", orient="records")

# AIThreat cruzado con AIAgents
threat_agents = (
    df[df["AIAgents"].notna() & df["AIThreat"].notna()]
    .groupby(["AIAgents", "AIThreat"])
    .size()
    .reset_index(name="count")
)
threat_agents.to_json("../docs/data/p2_threat_agents.json", orient="records")

# P3 ¿Qué herramientas, frameworks y LLMs específicos están usando los desarrolladores para
# construir y orquestar AI Agents, y cuáles son los más deseados para los próximos años?
def count_multivalor(series, top_n=15):
    return (
        series.dropna()
        .str.split(";")
        .explode()
        .str.strip()
        .value_counts()
        .head(top_n)
        .reset_index()
        .rename(columns={"index": "tool", "count": "count"})
    )

models = count_multivalor(df["AIModelsHaveWorkedWith"])
models.columns = ["tool", "count"]
models.to_json("../docs/data/p3_models.json", orient="records")

orchestration = count_multivalor(df["AIAgentOrchestration"])
orchestration.columns = ["tool", "count"]
orchestration.to_json("../docs/data/p3_orchestration.json", orient="records")

external = count_multivalor(df["AIAgentExternal"])
external.columns = ["tool", "count"]
external.to_json("../docs/data/p3_external.json", orient="records")

# P4 ¿Cuál es la brecha salarial entre regiones geográficas una vez normalizado el salario por PPP, y
# cómo se relaciona con el nivel de adopción de IA y el perfil del desarrollador?
salary_country = (
    df[df["ConvertedCompYearly"].notna() & df["Country"].notna()]
    .query("ConvertedCompYearly > 5000 and ConvertedCompYearly < 500000")
    .groupby("Country")["ConvertedCompYearly"]
    .agg(["median", "count"])
    .reset_index()
    .rename(columns={"median": "salary_median_usd", "count": "n"})
    .query("n >= 30")
    .sort_values("salary_median_usd", ascending=False)
)
salary_country["salary_median_usd"] = salary_country["salary_median_usd"].round(0)
salary_country.to_json("../docs/data/p4_salary_country.json", orient="records")

# P5 ¿Qué habilidades creen los desarrolladores que seguirán siendo valiosas en un mundo con IA
# avanzada (AIOpen), y difieren estas percepciones entre quienes ya usan AI Agents y quienes no?
stopwords = {"and", "to", "the", "of", "a", "in", "for", "is", "that", "will",
             "be", "with", "i", "it", "my", "are", "have", "as", "not", "on",
             "but", "or", "an", "at", "still", "also", "more", "can", "would",
             "which", "this", "by", "we", "skills", "skill", "ai", "human"}

words = (
    df["AIOpen"].dropna()
    .str.lower()
    .str.replace(r"[^a-z\s]", " ", regex=True)
    .str.split()
    .explode()
)
words = words[~words.isin(stopwords) & (words.str.len() > 3)]
freq = Counter(words).most_common(80)
word_freq = [{"word": w, "count": c} for w, c in freq]
with open("../docs/data/p5_aiopen_words.json", "w") as f:
    json.dump(word_freq, f)

print("Todos los JSON generados correctamente en docs/data")
print(os.listdir("../docs/data"))

Todos los JSON generados correctamente en docs/data
['p1_adoption_age.json', 'p1_adoption_country.json', 'p1_adoption_devtype.json', 'p1_adoption_dist.json', 'p2_aithreat.json', 'p2_jobsat_agents.json', 'p2_jobsat_factors.json', 'p2_jobsat_factors_split.json', 'p2_newrole.json', 'p2_threat_agents.json', 'p3_external.json', 'p3_models.json', 'p3_orchestration.json', 'p4_salary_adoption.json', 'p4_salary_country.json', 'p5_aiopen_words.json']


In [10]:
# P2 - JobSat factors ranking
jobsat_cols = {
    "JobSatPoints_1": "Control over quality",
    "JobSatPoints_2": "Autonomy and trust",
    "JobSatPoints_3": "Team collaboration",
    "JobSatPoints_4": "Expert mentors",
    "JobSatPoints_5": "Mentoring others",
    "JobSatPoints_6": "Developing expertise",
    "JobSatPoints_7": "Solving real problems",
    "JobSatPoints_8": "Job stability",
    "JobSatPoints_9": "Complex problem solving",
    "JobSatPoints_10": "New technologies",
    "JobSatPoints_11": "Competitive pay",
    "JobSatPoints_13": "Peer recognition",
    "JobSatPoints_14": "Leadership recognition",
    "JobSatPoints_16": "You like your manager",
}

for col in jobsat_cols.keys():
    df[col] = pd.to_numeric(df[col], errors="coerce")

means = df[list(jobsat_cols.keys())].mean().reset_index()
means.columns = ["col", "mean_rank"]
means["factor"] = means["col"].map(jobsat_cols)
means = means.sort_values("mean_rank")
means[["factor", "mean_rank"]].to_json("../docs/data/p2_jobsat_factors.json", orient="records")

agents_factors = (
    df[df["AIAgents"].notna()]
    .assign(uses_agents=df["AIAgents"].str.startswith("Yes"))
    .groupby("uses_agents")[list(jobsat_cols.keys())]
    .mean()
    .T
    .reset_index()
    .rename(columns={"index": "col", False: "no_agents", True: "yes_agents"})
)
agents_factors["factor"] = agents_factors["col"].map(jobsat_cols)
agents_factors = agents_factors.sort_values("yes_agents")
agents_factors[["factor", "yes_agents", "no_agents"]].to_json(
    "../docs/data/p2_jobsat_factors_split.json", orient="records"
)

print("Top 5 factores globales:")
print(means[["factor","mean_rank"]].head(5).to_string())
print("\nTop 5 factores usuarios AI Agents:")
print(agents_factors[["factor","yes_agents"]].head(5).to_string())


Top 5 factores globales:
                     factor  mean_rank
1        Autonomy and trust   5.216173
10          Competitive pay   5.703283
6     Solving real problems   5.884096
8   Complex problem solving   6.744658
0      Control over quality   6.867738

Top 5 factores usuarios AI Agents:
uses_agents                   factor  yes_agents
1                 Autonomy and trust    5.387955
10                   Competitive pay    5.608824
6              Solving real problems    5.895658
8            Complex problem solving    6.801821
0               Control over quality    6.995238


In [11]:
# Diferencias entre grupos
agents_factors["diff"] = agents_factors["yes_agents"] - agents_factors["no_agents"]
print(agents_factors[["factor", "yes_agents", "no_agents", "diff"]].sort_values("diff", ascending=False).to_string())

uses_agents                   factor  yes_agents  no_agents      diff
1                 Autonomy and trust    5.387955   5.009725  0.378230
0               Control over quality    6.995238   6.754191  0.241047
7                      Job stability    7.533193   7.296673  0.236520
13             You like your manager    7.969748   7.771273  0.198475
2                 Team collaboration    7.549020   7.403903  0.145117
8            Complex problem solving    6.801821   6.666539  0.135282
11                  Peer recognition    8.667647   8.552591  0.115056
6              Solving real problems    5.895658   5.831094  0.064564
5               Developing expertise    8.171008   8.236148 -0.065140
12            Leadership recognition    8.663866   8.736020 -0.072155
10                   Competitive pay    5.608824   5.685093 -0.076269
3                     Expert mentors    9.396499   9.701983 -0.305485
4                   Mentoring others    9.603361   9.954894 -0.351533
9                   

In [12]:
agents_factors["diff"] = agents_factors["yes_agents"] - agents_factors["no_agents"]
agents_factors[["factor", "yes_agents", "no_agents", "diff"]]\
    .sort_values("diff")\
    .to_json("../docs/data/p2_jobsat_factors_split.json", orient="records")
print("JSON actualizado")

JSON actualizado


In [13]:
# P1
adoption_age = (
    df[df["AIAgents"].notna() & df["Age"].notna()]
    .groupby("Age")["AIAgents"]
    .apply(lambda x: (x.str.startswith("Yes").sum() / len(x) * 100).round(1))
    .reset_index()
    .rename(columns={"AIAgents": "adoption_pct"})
)
print(adoption_age)
adoption_age.to_json("../docs/data/p1_adoption_age.json", orient="records")

                 Age  adoption_pct
0    18-24 years old          29.1
1    25-34 years old          32.3
2    35-44 years old          33.1
3    45-54 years old          30.5
4    55-64 years old          24.6
5  65 years or older          20.6
6  Prefer not to say          21.8


In [14]:
# P4 - Salario + adopción IA por país
salary_adoption = (
    df[df["ConvertedCompYearly"].notna() & df["Country"].notna() & df["AIAgents"].notna()]
    .query("ConvertedCompYearly > 5000 and ConvertedCompYearly < 500000")
    .groupby("Country")
    .agg(
        salary_median=("ConvertedCompYearly", "median"),
        adoption_pct=("AIAgents", lambda x: (x.str.startswith("Yes").sum() / len(x) * 100).round(1)),
        n=("ConvertedCompYearly", "count")
    )
    .reset_index()
    .query("n >= 30")
    .sort_values("salary_median", ascending=False)
)
salary_adoption["salary_median"] = salary_adoption["salary_median"].round(0)
salary_adoption.to_json("../docs/data/p4_salary_adoption.json", orient="records")
print(salary_adoption.head(10).to_string())

                                                  Country  salary_median  adoption_pct     n
147                              United States of America       150000.0          28.1  4777
62                                                 Israel       141188.0          39.6   154
133                                           Switzerland       140608.0          24.7   392
60                                                Ireland       116015.0          38.6   145
121                                             Singapore       103498.0          39.7    58
35                                                Denmark        98289.0          23.1   212
7                                               Australia        97514.0          28.7   534
145  United Kingdom of Great Britain and Northern Ireland        92576.0          23.2  1385
100                                                Norway        88945.0          33.0   176
23                                                 Canada        87550